In [1]:
!pip install datasets pandas -q

In [2]:
import pandas as pd
from datasets import load_dataset

In [3]:
ds = load_dataset("google/code_x_glue_cc_defect_detection")
df = pd.DataFrame(ds['train'])
print("Loaded rows:", len(df))
print(df.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/2.21M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21854 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2732 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2732 [00:00<?, ? examples/s]

Loaded rows: 21854
   id                                               func  target project  \
0   0  static av_cold int vdadec_init(AVCodecContext ...   False  FFmpeg   
1   1  static int transcode(AVFormatContext **output_...   False  FFmpeg   
2   2  static void v4l2_free_buffer(void *opaque, uin...   False  FFmpeg   
3   4  int av_opencl_buffer_write(cl_mem dst_cl_buf, ...   False  FFmpeg   
4   5  static int r3d_read_rdvo(AVFormatContext *s, A...    True  FFmpeg   

                                  commit_id  
0  973b1a6b9070e2bf17d17568cbaf4043ce931f51  
1  321b2a9ded0468670b7678b7c098886930ae16b2  
2  5d5de3eba4c7890c2e8077f5b4ae569671d11cf8  
3  57d77b3963ce1023eaf5ada8cba58b9379405cc8  
4  aba232cfa9b193604ed98f3fa505378d006b1b3b  


In [4]:
# remove duplicate code
df = df.drop_duplicates(subset=['func'])

# remove very short code snippets (less than 3 lines)
df = df[df['func'].apply(lambda x: len(x.strip().split('\n')) >= 3)]

# rename columns to simple names
df = df.rename(columns={'func': 'code', 'target': 'label'})

print("Rows after cleaning:", len(df))
print("Vulnerable vs Clean:")
print(df['label'].value_counts())

Rows after cleaning: 21802
Vulnerable vs Clean:
label
False    11819
True      9983
Name: count, dtype: int64


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os

os.makedirs('/content/drive/MyDrive/PolyGuard/01-data', exist_ok=True)

df.to_csv('/content/drive/MyDrive/PolyGuard/01-data/train.csv', index=False)
print("Saved successfully! Total rows:", len(df))

Saved successfully! Total rows: 21802


In [7]:
test = pd.read_csv('/content/drive/MyDrive/PolyGuard/01-data/train.csv')
print("Verification — shape:", test.shape)
print(test.head())

Verification — shape: (21802, 5)
   id                                               code  label project  \
0   0  static av_cold int vdadec_init(AVCodecContext ...  False  FFmpeg   
1   1  static int transcode(AVFormatContext **output_...  False  FFmpeg   
2   2  static void v4l2_free_buffer(void *opaque, uin...  False  FFmpeg   
3   4  int av_opencl_buffer_write(cl_mem dst_cl_buf, ...  False  FFmpeg   
4   5  static int r3d_read_rdvo(AVFormatContext *s, A...   True  FFmpeg   

                                  commit_id  
0  973b1a6b9070e2bf17d17568cbaf4043ce931f51  
1  321b2a9ded0468670b7678b7c098886930ae16b2  
2  5d5de3eba4c7890c2e8077f5b4ae569671d11cf8  
3  57d77b3963ce1023eaf5ada8cba58b9379405cc8  
4  aba232cfa9b193604ed98f3fa505378d006b1b3b  
